# 03 - 1D ResNet on raw windows, ECG-only vs +pulsatile, ONNX export (Colab, GPU)

Input `signals (N, 4, 2500)` + `mask (N, 4)` from `vtac.preprocess` (slots `ECG_1, ECG_2, PPG, ABP`).
The network multiplies signals by the mask and also feeds the mask as extra channels and into the
classifier head, so absent channels are explicit. Runs train on **train**, early-stop and tune the
threshold on **val**; the single exported model is evaluated on **test once**.

Ablations (see `docs/modeling.md`): C0-C4 model/imbalance/augmentation and K0-K3 channel sets
(ECG-only / +PPG / +ABP / +both). Re-run the notebook with `OFFSET = 0.0` for window ablation W0.

> Status: written but **not yet executed**. Use a GPU runtime. Expect to adjust epochs/width after
> the first run.

In [ ]:
# ---- Setup: mount Drive, install vtac from GitHub (pinned to a ref) -------------------------
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

VTAC_REF = "main"  # replace with a tag/commit once stable (reproducibility!)
!pip -q install "vtac[train] @ git+https://github.com/siddhu-exe/vtac-alarm-monitor.git@{VTAC_REF}"

import vtac
from vtac.provenance import installed_git_ref
GIT_REF = installed_git_ref(VTAC_REF)
print("vtac", vtac.__version__, "ref", GIT_REF)

DRIVE = Path("/content/drive/MyDrive/vtac")
for sub in ("processed", "replay", "models", "reports"):
    (DRIVE / sub).mkdir(parents=True, exist_ok=True)
SEED = 42

In [ ]:
import json, time, copy
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from vtac.config import VtacConfig
from vtac.dataset import load_windows
from vtac import metrics as M

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device", DEVICE, "| torch", torch.__version__)

OFFSET = 2.0  # 2.0 = project default (W1); rerun with 0.0 for the benchmark window (W0)
z = load_windows(DRIVE / "processed" / f"windows_off{OFFSET}.npz")
cfg = VtacConfig.from_dict(json.loads(str(z["config_json"])))
S = torch.tensor(z["signals"]); Mk = torch.tensor(z["mask"])
y = z["y"].astype(np.float32); split = z["split"]; patient = z["patient_id"]
idx = {s: np.flatnonzero(split == s) for s in ("train", "val", "test")}
SLOTS = list(cfg.channel_slots)
print({k: (len(v), round(float(y[v].mean()), 3)) for k, v in idx.items()}, S.shape)

## 1. Model

In [ ]:
class Block(nn.Module):
    def __init__(self, cin, cout, stride, k=7):
        super().__init__()
        self.c1 = nn.Conv1d(cin, cout, k, stride, k // 2, bias=False); self.b1 = nn.BatchNorm1d(cout)
        self.c2 = nn.Conv1d(cout, cout, k, 1, k // 2, bias=False); self.b2 = nn.BatchNorm1d(cout)
        self.sc = None if (cin == cout and stride == 1) else nn.Sequential(
            nn.Conv1d(cin, cout, 1, stride, bias=False), nn.BatchNorm1d(cout))

    def forward(self, x):
        o = F.relu(self.b1(self.c1(x)))
        o = self.b2(self.c2(o))
        return F.relu(o + (x if self.sc is None else self.sc(x)))


class ResNet1D(nn.Module):
    """(signals (B,C,T), mask (B,C)) -> logit (B,1). Absent channels are explicit via the mask."""

    def __init__(self, n_slots=4, width=32, dropout=0.3):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(2 * n_slots, width, 15, 2, 7, bias=False), nn.BatchNorm1d(width), nn.ReLU())
        self.layers = nn.Sequential(
            Block(width, width, 1), Block(width, 2 * width, 2), Block(2 * width, 2 * width, 1),
            Block(2 * width, 4 * width, 2), Block(4 * width, 4 * width, 1))
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(4 * width + n_slots, 1)

    def forward(self, signals, mask):
        m = mask.unsqueeze(-1)
        x = torch.cat([signals * m, m.expand(-1, -1, signals.shape[-1])], dim=1)
        h = self.layers(self.stem(x)).mean(-1)
        return self.fc(torch.cat([self.drop(h), mask], dim=1))


def focal_loss(logit, target, alpha, gamma=2.0):
    bce = F.binary_cross_entropy_with_logits(logit, target, reduction="none")
    p_t = torch.exp(-bce)
    a_t = alpha * target + (1 - alpha) * (1 - target)
    return (a_t * (1 - p_t) ** gamma * bce).mean()


@torch.no_grad()
def predict(model, X, Mask, bs=256):
    model.eval(); out = []
    for i in range(0, len(X), bs):
        out.append(torch.sigmoid(model(X[i:i + bs].to(DEVICE), Mask[i:i + bs].to(DEVICE))).squeeze(1).cpu())
    return torch.cat(out).numpy()

## 2. Training loop

- `imbalance="sampler"`: each epoch draws `len(train)` indices with inverse-class-frequency weights;
  `imbalance="focal"`: focal loss with alpha = 1 - prevalence.
- `dropout_aug`: per-sample probability of dropping each non-`ECG_1` channel (signal and mask) - trains
  the network for missing channels.
- `use`: slots allowed into the model (others are masked out for train, val and test) = channel ablation.
- Early stopping on validation PR-AUC.

In [ ]:
def slot_vector(use):
    return torch.tensor([1.0 if s in use else 0.0 for s in SLOTS])


def train_run(name, use, imbalance, dropout_aug=0.0, epochs=30, lr=1e-3, bs=64, patience=7, width=32, seed=SEED):
    torch.manual_seed(seed); np.random.seed(seed)
    sv = slot_vector(use)
    Xtr, Mtr = S[idx["train"]].to(DEVICE), (Mk[idx["train"]] * sv).to(DEVICE)
    ytr = torch.tensor(y[idx["train"]]).to(DEVICE)
    Xva, Mva, yva = S[idx["val"]], Mk[idx["val"]] * sv, y[idx["val"]]
    pos = float(ytr.mean()); n = len(ytr)
    w = torch.where(ytr > 0.5, 1 / pos, 1 / (1 - pos))
    model = ResNet1D(len(SLOTS), width).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * ((n + bs - 1) // bs))
    best, best_state, bad, hist = -1.0, None, 0, []
    for ep in range(epochs):
        model.train()
        order = torch.multinomial(w, n, replacement=True) if imbalance == "sampler" else torch.randperm(n, device=DEVICE)
        tot = 0.0
        for i in range(0, n, bs):
            b = order[i:i + bs]
            xb, mb, yb = Xtr[b], Mtr[b], ytr[b].unsqueeze(1)
            if dropout_aug > 0:
                drop = (torch.rand_like(mb) < dropout_aug)
                drop[:, 0] = False  # always keep ECG_1
                mb = mb * (~drop).float()
            logit = model(xb, mb)
            loss = focal_loss(logit, yb, alpha=1 - pos) if imbalance == "focal" else F.binary_cross_entropy_with_logits(logit, yb)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
            tot += loss.item() * len(b)
        p_va = predict(model, Xva, Mva)
        pr = M.pr_auc(yva, p_va)
        hist.append(dict(epoch=ep, loss=tot / n, val_pr_auc=pr, val_roc_auc=M.roc_auc(yva, p_va)))
        if pr > best:
            best, best_state, bad = pr, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= patience:
                break
    model.load_state_dict(best_state)
    p_va = predict(model, Xva, Mva)
    tuned = M.tune_thresholds(yva, p_va, tpr_target=cfg.tpr_target)
    print(f"{name:22s} val PR-AUC {best:.3f}  ROC-AUC {M.roc_auc(yva, p_va):.3f}  | constrained thr {tuned['constrained']['threshold']:.3f} "
          f"TPR {tuned['constrained']['tpr']:.3f} PPV {tuned['constrained']['ppv']:.3f} score {tuned['constrained']['challenge_score']:.3f}  ({ep+1} ep)")
    return dict(name=name, use=list(use), imbalance=imbalance, dropout_aug=dropout_aug, model=model.cpu(), tuned=tuned,
                val_pr_auc=best, hist=hist, sv=sv)

## 3. Ablation runs (validation numbers only)

C0-C3: ECG-only vs all channels, weighted sampler vs focal loss. C4: best imbalance strategy on all
channels + channel-dropout augmentation. K1/K2: ECG+PPG, ECG+ABP (K0 = C0/C1, K3 = C2/C3).

In [ ]:
ECG = ("ECG_1", "ECG_2"); ALL = tuple(SLOTS)
runs = {}
for name, use, imb in [("C0_ecg_sampler", ECG, "sampler"), ("C1_ecg_focal", ECG, "focal"),
                       ("C2_all_sampler", ALL, "sampler"), ("C3_all_focal", ALL, "focal")]:
    runs[name] = train_run(name, use, imb)
best_imb = max(("sampler", "focal"), key=lambda i: runs[f"C{2 if i == 'sampler' else 3}_all_{i}"]["val_pr_auc"])
print("best imbalance strategy on all channels:", best_imb)
runs["C4_all_dropout"] = train_run("C4_all_dropout", ALL, best_imb, dropout_aug=0.15)
for name, use in [("K1_ecg_ppg", ECG + ("PPG",)), ("K2_ecg_abp", ECG + ("ABP",))]:
    runs[name] = train_run(name, use, best_imb)

In [ ]:
table = pd.DataFrame([dict(run=r["name"], channels="+".join(r["use"]), imbalance=r["imbalance"], aug=r["dropout_aug"],
                           val_pr_auc=r["val_pr_auc"], val_roc_auc=r["tuned"]["constrained"]["roc_auc"],
                           thr_c=r["tuned"]["constrained"]["threshold"], tpr_c=r["tuned"]["constrained"]["tpr"],
                           ppv_c=r["tuned"]["constrained"]["ppv"], score_c=r["tuned"]["constrained"]["challenge_score"],
                           score_u=r["tuned"]["unconstrained"]["challenge_score"]) for r in runs.values()]).round(4)
table.to_csv(DRIVE / "reports" / f"cnn_val_ablation_off{OFFSET}.csv", index=False)
table

## 4. Choose and export the final model

Final = best validation PR-AUC among runs that use ECG + both pulsatile slots (C2/C3/C4). Then:
test evaluation **once** with the validation-tuned thresholds, ONNX export, parity check, and
`model_config.json` (both thresholds stored; `threshold` = the TPR-constrained one).

In [ ]:
cands = [r for r in runs.values() if set(r["use"]) == set(ALL)]
final = max(cands, key=lambda r: r["val_pr_auc"])
print("FINAL RUN:", final["name"], "val PR-AUC", round(final["val_pr_auc"], 4))

sv = final["sv"]
Xte, Mte, yte = S[idx["test"]], Mk[idx["test"]] * sv, y[idx["test"]]
final["model"].to(DEVICE)
p_te = predict(final["model"], Xte, Mte); final["model"].cpu()
test = {}
for kind in ("constrained", "unconstrained"):
    thr = final["tuned"][kind]["threshold"]
    ev = M.evaluate(yte, p_te, thr)
    ev["challenge_ci95"] = M.group_bootstrap_ci(lambda a, b, t=thr: M.evaluate(a, b, t)["challenge_score"], yte, p_te, patient[idx["test"]], n_boot=500, seed=SEED)
    test[kind] = ev
    print(f"TEST {kind:13s} thr {thr:.3f}  TPR {ev['tpr']:.3f}  PPV {ev['ppv']:.3f}  TNR {ev['tnr']:.3f}  PR-AUC {ev['pr_auc']:.3f}  "
          f"ROC-AUC {ev['roc_auc']:.3f}  score {ev['challenge_score']:.3f} {tuple(round(v, 3) for v in ev['challenge_ci95'])}  FN {ev['fn']} FP {ev['fp']}")
has_puls = ((Mk[idx["test"]][:, 2] > 0) | (Mk[idx["test"]][:, 3] > 0)).numpy()
for label, m in (("with_pulsatile", has_puls), ("ecg_only", ~has_puls)):
    if m.sum() > 10 and 0 < yte[m].sum() < m.sum():
        print("  test", label, "n", int(m.sum()), "PR-AUC", round(M.pr_auc(yte[m], p_te[m]), 3))

In [ ]:
# ---- ONNX export (legacy TorchScript exporter for a stable, simple graph) --------------------
model = final["model"].eval().cpu()
dummy = (torch.randn(2, len(SLOTS), cfg.window_samples), torch.ones(2, len(SLOTS)))
onnx_path = DRIVE / "models" / "model.onnx"
kw = dict(input_names=["signals", "mask"], output_names=["logit"], opset_version=17,
          dynamic_axes={"signals": {0: "N"}, "mask": {0: "N"}, "logit": {0: "N"}})
try:
    torch.onnx.export(model, dummy, str(onnx_path), dynamo=False, **kw)
except TypeError:  # older torch without the dynamo flag
    torch.onnx.export(model, dummy, str(onnx_path), **kw)

import onnxruntime as ort
sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
xb, mb = Xte[:64].numpy(), Mte[:64].numpy()
with torch.no_grad():
    ref = model(torch.tensor(xb), torch.tensor(mb)).numpy()
got = sess.run(["logit"], {"signals": xb, "mask": mb})[0]
print("max |torch - onnx| logit diff:", float(np.abs(ref - got).max()))
assert np.abs(ref - got).max() < 1e-3
one = sess.run(["logit"], {"signals": xb[:1], "mask": mb[:1]})[0]  # dynamic batch = 1
assert one.shape == (1, 1)
t0 = time.time(); [sess.run(["logit"], {"signals": xb[:1], "mask": mb[:1]}) for _ in range(50)]
print(f"onnxruntime CPU single-window latency ~{(time.time()-t0)/50*1e3:.1f} ms (Colab CPU; laptop will differ)")

In [ ]:
thr_c, thr_u = final["tuned"]["constrained"]["threshold"], final["tuned"]["unconstrained"]["threshold"]
clean = lambda d: json.loads(json.dumps(d, default=float))
cfg_final = cfg.with_updates(
    threshold=thr_c, threshold_constrained=thr_c, threshold_unconstrained=thr_u,
    model_version=f"resnet1d-{final['name']}-off{OFFSET}", vtac_git_ref=GIT_REF,
    extra=clean(dict(final_run=final["name"], imbalance=final["imbalance"], dropout_aug=final["dropout_aug"], seed=SEED,
                     val=dict(constrained=final["tuned"]["constrained"], unconstrained=final["tuned"]["unconstrained"]),
                     test=test)))
cfg_final.save(DRIVE / "models" / "model_config.json")
print(json.dumps({k: v for k, v in cfg_final.to_dict().items() if k != "extra"}, indent=1)[:900])
print("saved model.onnx + model_config.json to", DRIVE / "models")

## 5. Parity batch (export only; no retraining)

Saves ~32 fixed test windows (all mask patterns included) with their PyTorch scores to
`models/parity_batch.npz`. Copy it next to `model.onnx`; `tests/test_onnx_parity.py` then checks
ONNX vs PyTorch scores with max abs diff < 1e-4. Also saves `model_state.pt` (weights were not
saved before).

In [ ]:
# ---- Parity batch: ~32 fixed test windows + PyTorch scores, for the repo's ONNX-vs-PyTorch test ----
# EXPORT-ONLY cell, no training. Run it in the SAME Colab session that trained `final`
# (it needs final, Xte, Mte, yte, idx, sess, cfg_final, DRIVE, SEED from the cells above).
assert "final" in globals() and "cfg_final" in globals(), \
    "No trained model in this runtime (weights are not on Drive). Do NOT retrain here (it would change the model); weights exist only in the training session."
rng = np.random.default_rng(SEED)
mte = Mte.numpy()
by_pattern = {}
for r, row in enumerate(mte):
    by_pattern.setdefault(tuple(int(v) for v in row), []).append(r)
N_PARITY = 32
chosen = [int(rng.choice(rows)) for rows in by_pattern.values()][:N_PARITY]  # every mask pattern once
rest = [i for i in rng.permutation(len(mte)).tolist() if i not in set(chosen)]
rest_pos = [i for i in rest if yte[i] > 0.5]
rest_neg = [i for i in rest if yte[i] <= 0.5]
need = N_PARITY - len(chosen)
n_pos = max(0, min(len(rest_pos), N_PARITY // 2 - int(yte[chosen].sum()), need))
chosen = sorted(chosen + rest_pos[:n_pos] + rest_neg[:need - n_pos])
print("mask patterns in test:", {k: len(v) for k, v in by_pattern.items()})
print("parity batch:", len(chosen), "windows,", int(yte[chosen].sum()), "true;",
      "masked-channel windows:", int((mte[chosen].min(axis=1) == 0).sum()))

xb = Xte[chosen].numpy().astype(np.float32)
mb = mte[chosen].astype(np.float32)
model = final["model"].eval().cpu()
with torch.no_grad():
    logit_t = model(torch.tensor(xb), torch.tensor(mb)).numpy()[:, 0]
p_t = 1.0 / (1.0 + np.exp(-logit_t.astype(np.float64)))
logit_o = sess.run(["logit"], {"signals": xb, "mask": mb})[0][:, 0]
p_o = 1.0 / (1.0 + np.exp(-logit_o.astype(np.float64)))
print("max |logit diff|", float(np.abs(logit_t - logit_o).max()), "| max |p diff|", float(np.abs(p_t - p_o).max()))
assert np.abs(p_t - p_o).max() < 1e-4

np.savez_compressed(
    DRIVE / "models" / "parity_batch.npz",
    signals=xb, mask=mb, y=yte[chosen].astype(np.int8),
    test_index=idx["test"][chosen].astype(np.int64),
    torch_logit=logit_t.astype(np.float32), torch_p=p_t,
    threshold=np.float64(cfg_final.threshold), model_version=np.array(cfg_final.model_version),
)
torch.save(final["model"].state_dict(), DRIVE / "models" / "model_state.pt")  # so parity can be redone later
print("saved parity_batch.npz + model_state.pt to", DRIVE / "models")


## Next

Download `models/model.onnx` and `models/model_config.json` into the repo's `models/` (gitignored).
The backend ONNX loader (`VTAC_MODEL_MODE=onnx`) reads them. Record the ablation tables in
`docs/modeling.md` and the W0/W1 outcome in `docs/decisions.md`.